In [0]:
%sql
CREATE CATALOG IF NOT EXISTS machine_health;
CREATE SCHEMA  IF NOT EXISTS machine_health.prod;
CREATE SCHEMA  IF NOT EXISTS machine_health.dev;
USE machine_health.prod;

CREATE OR REPLACE TABLE sensor_readings_raw (
  sensor_id STRING, machine_type STRING, reading_time TIMESTAMP, axis STRING,
  peak_accel_g DOUBLE, vibration_rms_mm_s DOUBLE, temp_c DOUBLE);

CREATE OR REPLACE TABLE ingest_watermark (
  machine_type STRING, last_reading_time TIMESTAMP);

CREATE OR REPLACE TABLE predictions_latest (
  sensor_id STRING, machine_type STRING, reading_time TIMESTAMP, axis STRING,
  peak_accel_g DOUBLE, vibration_rms_mm_s DOUBLE, temp_c DOUBLE,
  is_healthy BIGINT, model_tag STRING, run_id STRING);

CREATE OR REPLACE TABLE predictions_history (
  sensor_id STRING, machine_type STRING, reading_time TIMESTAMP, axis STRING,
  peak_accel_g DOUBLE, vibration_rms_mm_s DOUBLE, temp_c DOUBLE,
  is_healthy BIGINT, model_tag STRING, run_id STRING);

CREATE OR REPLACE TABLE health_decisions (
  sensor_id STRING, machine_type STRING, reading_time TIMESTAMP, axis STRING,
  is_healthy BIGINT, model_tag STRING, decision_note STRING)
  TBLPROPERTIES (delta.enableDeletionVectors = false);

CREATE OR REPLACE TABLE alert_candidates (
  issue_id STRING, sensor_id STRING, machine_type STRING, reading_time TIMESTAMP, axis STRING,
  is_healthy BIGINT, model_tag STRING, decision_note STRING, status STRING, removal_reason STRING,
  mlflow_run_id STRING, job_id STRING, job_run_id STRING);

CREATE OR REPLACE TABLE maintenance_advice (
  sensor_id STRING, machine_type STRING, reading_time TIMESTAMP,
  category STRING, advice STRING, details STRING);

CREATE OR REPLACE TABLE thermal_advice (
  sensor_id STRING, machine_type STRING, reading_time TIMESTAMP, advice STRING);

CREATE OR REPLACE TABLE pipeline_run_audit (
  job_id STRING, job_run_id STRING, task_run_id STRING, task_key STRING,
  machine_type STRING, attempt_number INT, status STRING,
  start_time TIMESTAMP, end_time TIMESTAMP, cluster_id STRING, error_message STRING,
  run_page_url STRING, task_run_page_url STRING,
  table_name STRING, operation STRING,
  conflicting_job_run_id STRING, conflicting_run_page_url STRING,
  read_version BIGINT, conflicting_version BIGINT);

CREATE TABLE IF NOT EXISTS pipeline_cost_monitoring (
  job_id STRING, job_name STRING, run_id STRING, sku_name STRING,
  usage_quantity DECIMAL(38,18), usage_unit STRING, estimated_cost_usd DECIMAL(38,6),
  first_usage TIMESTAMP, last_usage TIMESTAMP, usage_date DATE, ingested_at TIMESTAMP)
  PARTITIONED BY (usage_date)
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
  );

CREATE TABLE IF NOT EXISTS pipeline_cost_summary (
  job_id STRING, job_name STRING,
  total_usage DECIMAL(38,18), estimated_total_cost_usd DECIMAL(38,6),
  run_count BIGINT, earliest_usage TIMESTAMP, latest_usage TIMESTAMP);
